# 词袋模型  
该部分以 IMDB 影评情感分类任务为案例，讲解词袋 (Bag‑of‑Words) 实现文本情感分析完整流程。基于 IMDB 影评，判断影评情感是正面（1）还是负面（0）。训练集提供影评文本 + 真实情感标签；测试集只有影评文本，需要模型预测情感。



In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


## 读取数据集  
通常使用pandas读取数据文件，import引入pandas包，利用.read_csv函数读取


In [4]:
import pandas as pd

train = pd.read_csv(
    "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip",
    header=0,
    delimiter="\t",
    quoting=3
)

print (train.shape)
print (train.columns.to_numpy())
print (train["review"][0])

(25000, 3)
['id' 'sentiment' 'review']
"With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.<br /><br />Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.<br /><br />The actu

`header=0`表示文件的第一行包含列名,`delimiter="\t"`表示字段之间用分隔符分割,`quoting=3`表示忽略重引号

## 文字预处理  
### 去除HTML标记：BeautifulSoup软件包
什么是HTML标签？
HTML标签是网页用来标记内容的标记符号，一般成对出现，格式：<标签名>内容</标签名>。告诉浏览器这一段是什么内容（标题、段落、图片、超链接等）。

In [5]:
#去除HTML标记
# 引入BeautifulSoup软件包
from bs4 import BeautifulSoup             

#初始化BeautifulSoup对象     
b = BeautifulSoup(train["review"][0])  

print (train["review"][0])
print (b.get_text())

"With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.<br /><br />Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.<br /><br />The actual feature film bit when it finally sta

### 处理标点符号和数字
我们采用正则化表达式工具包re中的方法re.sub()。   
正则化表示群成员身份 和 表示“非”即可。换句话说，上面的 re.sub（） 语句是：“找任何不是小写字母（）或大写字母（）的东西，然后用空格替换。

In [6]:
#处理标点和数字
import re

letters_only = re.sub("[^a-zA-Z]",      #非字母字符       
                      " ",              #空格替换    
                      b.get_text() )    
print (letters_only)

 With all this stuff going down at the moment with MJ i ve started listening to his music  watching the odd documentary here and there  watched The Wiz and watched Moonwalker again  Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent  Moonwalker is part biography  part feature film which i remember going to see at the cinema when it was originally released  Some of it has subtle messages about MJ s feeling towards the press and also the obvious message of drugs are bad m kay Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring  Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him The actual feature film bit when it finally starts is only on for    mi

### 全部转为小写并拆分位单个词

In [7]:
#大写字母转换成小写字母，并拆分成单个词（代币化）
lower_case = letters_only.lower()        #转换为小写字母
words = lower_case.split()               #分裂成单个单词
print (lower_case)
print (words)

 with all this stuff going down at the moment with mj i ve started listening to his music  watching the odd documentary here and there  watched the wiz and watched moonwalker again  maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent  moonwalker is part biography  part feature film which i remember going to see at the cinema when it was originally released  some of it has subtle messages about mj s feeling towards the press and also the obvious message of drugs are bad m kay visually impressive but of course this is all about michael jackson so unless you remotely like mj in anyway then you are going to hate this and find it boring  some may call mj an egotist for consenting to the making of this movie but mj and most of his fans would say that he made it for the fans which if true is really nice of him the actual feature film bit when it finally starts is only on for    mi

### 去除停止词  
最后，我们需要决定如何处理那些经常出现但意义不大的词。这些词被称为“停止词”;在英语中，它们包含了诸如“a”、“and”、“is”和“the”这样的词。方便的是，有些 Python 包内置了停止词列表。让我们从Python自然语言工具包（NLTK）导入一个停止词列表。

In [8]:
#停止词列表
from nltk.corpus import stopwords  #引入停止词列表
print (stopwords.words("english")) 

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [9]:
#去除影评中的停止词
words = [w for w in words if not  w in stopwords.words("english")]
print (words)


['stuff', 'going', 'moment', 'mj', 'started', 'listening', 'music', 'watching', 'odd', 'documentary', 'watched', 'wiz', 'watched', 'moonwalker', 'maybe', 'want', 'get', 'certain', 'insight', 'guy', 'thought', 'really', 'cool', 'eighties', 'maybe', 'make', 'mind', 'whether', 'guilty', 'innocent', 'moonwalker', 'part', 'biography', 'part', 'feature', 'film', 'remember', 'going', 'see', 'cinema', 'originally', 'released', 'subtle', 'messages', 'mj', 'feeling', 'towards', 'press', 'also', 'obvious', 'message', 'drugs', 'bad', 'kay', 'visually', 'impressive', 'course', 'michael', 'jackson', 'unless', 'remotely', 'like', 'mj', 'anyway', 'going', 'hate', 'find', 'boring', 'may', 'call', 'mj', 'egotist', 'consenting', 'making', 'movie', 'mj', 'fans', 'would', 'say', 'made', 'fans', 'true', 'really', 'nice', 'actual', 'feature', 'film', 'bit', 'finally', 'starts', 'minutes', 'excluding', 'smooth', 'criminal', 'sequence', 'joe', 'pesci', 'convincing', 'psychopathic', 'powerful', 'drug', 'lord', 

### 把一切整合起来  
现在我们有代码清理一条评审——但我们需要清理25,000条培训评审！为了让代码可复用，我们创建一个可以多次调用的函数：

In [10]:
#清理一条评论的方法，可反复利用
def review_to_words( raw_review ):
    # 将条影评转换为一个字符串
    
    # 1.去除HTMl标签
    review = BeautifulSoup(raw_review).get_text()
    
    # 2.除非字母字符      
    letter_only = re.sub("[^a-zA-Z]"," ",review)
    
    # 3.转换成小写字母，并分裂成单个单词
    lower_case = letter_only.lower()
    words = lower_case.split()
    
    # 4. 在 Python 中，在集合 (set) 中查找元素远快于列表 (list)，因此把停用词转换成集合。
    stops = set(stopwords.words("english"))                
    
    # 5.去除停止词
    words = [w for w in words if not w in stops]  
    
    # 6. 把单词重新拼接成一个字符串，单词之间用空格分隔，然后返回该结果。
    return( " ".join( words )) 
      

接下来让我们一次性清理所有数据集

In [11]:
#清理所有训练集

num_reviews = train["review"].size

clean_train_reviews = []
 
for i in range( 0, num_reviews ):
    clean_train_reviews.append( review_to_words( train["review"][i] ) )

## 从单词袋中创建特征（使用 scikit-learn）
既然我们的训练评审已经整理好，我们如何将它们转化为机器学习的某种数值表示？一种常见的方法叫做“词袋”。词袋模型从所有文档中学习词汇，然后通过计算每个单词出现的次数来建模每份文档。例如，考虑以下两句话：

句子一：“猫坐在帽子上”

句子二：“狗吃了猫和帽子”

从这两句话中，我们的词汇如下：

{ the， cat， sat， on， hat， dog， ate and }

为了获得我们的词袋，我们会统计每个词在每句话中出现的次数。在第一句中，“the”出现两次，“cat”、“sat”、“on”和“hat”各出现一次，因此第一句的特征向量为：

{ the， cat， sat， on， hat， dog， ate and }

句子1：{ 2， 1， 1， 1， 1， 0， 0， 0 }

同样，句子2的特征为：{3， 1， 0， 0， 1， 1， 1， 1}

在IMDB数据中，我们有大量评论，这会让我们拥有丰富的词汇量。为了限制特征向量的大小，我们应选择一个最大词汇量。下面，我们使用了5000个最常用的词（记住，停止词已经被移除）。

我们将使用scikit-learn的feature_extraction模块来创建单词包功能。

In [14]:
print ("Creating the bag of words...\n")
from sklearn.feature_extraction.text import CountVectorizer

# 初始化 CountVectorizer 对象，它是 scikit‑learn 库中实现词袋模型的工具。
# analyzer = "word" 以单词为分析单元
# stop_words = None 因为文本预处理时已经手动过滤停止词，所以设为无
# max_features = 5000 保留出现频率最高的5000个单词作为特征
vectorizer = CountVectorizer(analyzer = "word",   \
                             tokenizer = None,    \
                             preprocessor = None, \
                             stop_words = None,   \
                             max_features = 5000)   

train_data_features = vectorizer.fit_transform(clean_train_reviews)

# Numpy 数组使用起来很方便，因此将结果转换为数组。
train_data_features = train_data_features.toarray()
print (train_data_features.shape)

Creating the bag of words...

(25000, 5000)


### fit_transform  
拟合模型并学习词汇表  
将训练数据转换为特征向量。


In [15]:
# 查看词汇表中的单词
vocab = vectorizer.get_feature_names_out()
print (vocab)

['abandoned' 'abc' 'abilities' ... 'zombie' 'zombies' 'zone']


## 随机森林  
关于随机森林的具体内容，可查阅2026年7月份关于吴恩达机器学习课程的笔记整理

In [16]:
print ("Training the random forest...")
from sklearn.ensemble import RandomForestClassifier

# 初始化一个拥有 100 棵树的随机森林分类器
forest = RandomForestClassifier(n_estimators = 100) 

# 使用词袋作为特征、情感标签作为输出标签，在训练集上拟合随机森林模型。

forest = forest.fit( train_data_features, train["sentiment"] )

Training the random forest...


In [17]:
#读取测试集数据
test = pd.read_csv("/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip", header=0, delimiter="\t", \
                   quoting=3 )

print (test.shape)

num_reviews = len(test["review"])
clean_test_reviews = [] 

print ("Cleaning and parsing the test set movie reviews...\n")
for i in range(0,num_reviews):
    if( (i+1) % 1000 == 0 ):
        print ("Review %d of %d\n" % (i+1, num_reviews))
    clean_review = review_to_words( test["review"][i] )
    clean_test_reviews.append( clean_review )

test_data_features = vectorizer.transform(clean_test_reviews)
test_data_features = test_data_features.toarray()

#利用已经训练好的随机森林模型进行预测
result = forest.predict(test_data_features)

# 将预测结果复制到 pandas 数据表中，表包含id列与sentiment列。
output = pd.DataFrame( data={"id":test["id"], "sentiment":result} )

output.to_csv( "Bag_of_Words_model.csv", index=False, quoting=3 )
print(output.head(10))

(25000, 2)
Cleaning and parsing the test set movie reviews...

Review 1000 of 25000

Review 2000 of 25000

Review 3000 of 25000

Review 4000 of 25000

Review 5000 of 25000

Review 6000 of 25000

Review 7000 of 25000

Review 8000 of 25000

Review 9000 of 25000

Review 10000 of 25000

Review 11000 of 25000

Review 12000 of 25000

Review 13000 of 25000

Review 14000 of 25000

Review 15000 of 25000

Review 16000 of 25000

Review 17000 of 25000

Review 18000 of 25000

Review 19000 of 25000

Review 20000 of 25000

Review 21000 of 25000

Review 22000 of 25000

Review 23000 of 25000

Review 24000 of 25000

Review 25000 of 25000

           id  sentiment
0  "12311_10"          1
1    "8348_2"          0
2    "5828_4"          1
3    "7186_2"          1
4   "12128_7"          1
5    "2913_8"          0
6    "4396_1"          0
7     "395_2"          0
8   "10616_1"          0
9    "9074_9"          1
